# 1. Initializations

## 1.1 General imports

In [ ]:
### general
from itertools import islice

### data management
import pandas as pd
import numpy as np

### machine learning (scikit-learn)
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_absolute_error, r2_score

### graphical
import matplotlib.pyplot as plt
# for jupyter notebook management
%matplotlib inline
import seaborn as sns


## 1.2 General dataframe functions

In [ ]:
import smartcheck.preprocessing_project_specific as pps
print(dir(pps))

In [ ]:
import smartcheck.dataframe_common as dfc
import smartcheck.dataframe_project_specific as dfps

## 1.3 Specific preprocessing classes

In [ ]:
import smartcheck.preprocessing_project_specific as pps

# 2. Loading and Preprocessing

In [ ]:
df_cpt_raw = dfc.load_dataset_from_config('velo_comptage_ml_ready_data', sep=',', index_col=0)

if df_cpt_raw is not None and isinstance(df_cpt_raw, pd.DataFrame):
    df_cpt = df_cpt_raw.copy()

In [ ]:
df_cpt.info()

## 2.1 Preprocessing pipelines

In [ ]:
keep_cols = [
    "nom_du_site_de_comptage",
    "comptage_horaire",
    "date_et_heure_de_comptage",
    "orientation_compteur",
    "latitude",
    "longitude",
    "arrondissement",
    "jour_ferie",
    "vacances_scolaires",
    "temperature_2m_c",
    "rain_mm",
    "snowfall_cm",
    # "weather_code_wmo_code",
    "elevation",
    "weather_code_wmo_code_category",
]

pipe_preproc = Pipeline([
    ("filter_columns", pps.ColumnFilterTransformer(columns_to_keep=keep_cols)),
    ("add_datetime_features", pps.DatetimePeriodicsTransformer(timestamp_col="date_et_heure_de_comptage")),
])

df_preproc = pipe_preproc.fit_transform(df_cpt)
if df_preproc is not None and isinstance(df_preproc, pd.DataFrame):
    df = df_preproc.copy()

In [ ]:
# Verification des distributions après preprocessing
df.info()
display(df.select_dtypes(include=np.number).describe())
display(df.select_dtypes(include='object').describe())

#### Iterative manual feature selections with VIF

In [ ]:
# # VarianceInflationFactor (VIF)
# # - VIF > 5 → multicolinéarité forte
# # - VIF > 10 → à supprimer quasi sûr
# X = df.drop(columns='comptage_horaire').select_dtypes(include=np.number)
# X = X.astype("float64")
# vif_data = pd.DataFrame()
# vif_data["feature"] = X.columns
# vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
# print(vif_data)

# # Matrice de correlation 
# plt.figure(figsize=(20, 20))  # Taille de la figure
# corr_mat = X.corr()
# sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="coolwarm", mask=np.triu(corr_mat), center=0)
# plt.show()

>🔥 Corrélations proches de ±1 :
>
>| Couple de variables                                   | Corrélation | Action recommandée              |
>| ----------------------------------------------------- | ----------- | ------------------------------- |
>| `day_of_year` ↔ `day`                                 | **1.00**    | ❌ garder un seul des deux       |
>| `month` ↔ `sin_month` / `cos_month`                   | ±0.67/0.56  | ❌ garder `sin/cos`, pas `month` |
>| `week` ↔ `sin_week` / `cos_week`                      | ±0.97/0.93  | ❌ garder `sin/cos`, pas `week`  |
>| `hour` ↔ `sin_hour`                                   | **–0.78**   | ❌ idem, `sin/cos_hour` > `hour` |
>| `day_of_week` ↔ `sin_day_of_week` / `cos_day_of_week` | \~±0.75     | ❌ idem, `sin/cos` > brut        |


In [ ]:
# Ajustement itération 1
col_to_drop1 = [
    'date_et_heure_de_comptage_day_of_year',
    'date_et_heure_de_comptage_year',
    'date_et_heure_de_comptage_month',
    'date_et_heure_de_comptage_day',
    'date_et_heure_de_comptage_day_of_week',
    'date_et_heure_de_comptage_hour'
]
# X1 = X.drop(columns=col_to_drop1)

In [ ]:
# # VarianceInflationFactor (VIF)
# # - VIF > 5 → multicolinéarité forte
# # - VIF > 10 → à supprimer quasi sûr
# vif_data = pd.DataFrame()
# vif_data["feature"] = X1.columns
# vif_data["VIF"] = [variance_inflation_factor(X1.values, i) for i in range(X1.shape[1])]
# print(vif_data)

# # Matrice de correlation 
# plt.figure(figsize=(20, 20))  # Taille de la figure
# corr_mat = X1.corr()
# sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="coolwarm", mask=np.triu(corr_mat), center=0)
# plt.show()

In [ ]:
# Ajustement itération 2
col_to_drop2 = [
    'latitude',
    'longitude',
    'date_et_heure_de_comptage_sin_month',
    'date_et_heure_de_comptage_sin_week',
]
# X2 = X1.drop(columns=col_to_drop2)

In [ ]:
# # VarianceInflationFactor (VIF)
# # - VIF > 5 → multicolinéarité forte
# # - VIF > 10 → à supprimer quasi sûr
# vif_data = pd.DataFrame()
# vif_data["feature"] = X2.columns
# vif_data["VIF"] = [variance_inflation_factor(X2.values, i) for i in range(X2.shape[1])]
# print(vif_data)

# # Matrice de correlation 
# plt.figure(figsize=(20, 20))  # Taille de la figure
# corr_mat = X2.corr()
# sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="coolwarm", mask=np.triu(corr_mat), center=0)
# plt.show()

In [ ]:
# Ajustement itération 3
col_to_drop3 = [
    'date_et_heure_de_comptage_cos_month',
]
# X3 = X2.drop(columns=col_to_drop3)

In [ ]:
# # VarianceInflationFactor (VIF)
# # - VIF > 5 → multicolinéarité forte
# # - VIF > 10 → à supprimer quasi sûr
# vif_data = pd.DataFrame()
# vif_data["feature"] = X3.columns
# vif_data["VIF"] = [variance_inflation_factor(X3.values, i) for i in range(X3.shape[1])]
# print(vif_data)

# # Matrice de correlation 
# plt.figure(figsize=(20, 20))  # Taille de la figure
# corr_mat = X3.corr()
# sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="coolwarm", mask=np.triu(corr_mat), center=0)
# plt.show()

In [ ]:
df = df.drop(columns=col_to_drop1+col_to_drop2+col_to_drop3)
df_feat_sel = df.copy()

## 2.2 Column Transformers

#### Categorical and Numerical

In [ ]:
# === 1. Chargement des données ===
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer

import smartcheck.dataframe_common as dfc
import smartcheck.preprocessing_project_specific as pps

df_cpt_raw = dfc.load_dataset_from_config('velo_comptage_ml_ready_data', sep=',', index_col=0)

if df_cpt_raw is not None and isinstance(df_cpt_raw, pd.DataFrame):
    df_cpt = df_cpt_raw.copy()
    print("✅ Données brutes chargées avec succès.")
else:
    raise SystemExit("❌ Échec du chargement des données.")



In [ ]:
# === 2. Prétraitement initial avec pipeline smartcheck ===

keep_cols = [
    "nom_du_site_de_comptage",
    "comptage_horaire",
    "date_et_heure_de_comptage",
    "orientation_compteur",
    "latitude",
    "longitude",
    "arrondissement",
    "jour_ferie",
    "vacances_scolaires",
    "temperature_2m_c",
    "rain_mm",
    "snowfall_cm",
    "elevation",
    "weather_code_wmo_code_category",
]

pipe_preproc = Pipeline([
    ("filter_columns", pps.ColumnFilterTransformer(columns_to_keep=keep_cols)),
    ("add_datetime_features", pps.DatetimePeriodicsTransformer(timestamp_col="date_et_heure_de_comptage")),
])

df_preproc = pipe_preproc.fit_transform(df_cpt)

if df_preproc is not None and isinstance(df_preproc, pd.DataFrame):
    df = df_preproc.copy()
    print("✅ Pipeline smartcheck appliqué avec succès.")
else:
    raise SystemExit("❌ Échec du prétraitement initial avec smartcheck.")


In [ ]:
# Colonnes à conserver
keep_cols = [
    "nom_du_site_de_comptage",
    "comptage_horaire",
    "date_et_heure_de_comptage",
    "orientation_compteur",
    "latitude",
    "longitude",
    "arrondissement",
    "jour_ferie",
    "vacances_scolaires",
    "temperature_2m_c",
    "rain_mm",
    "snowfall_cm",
    "elevation",
    "weather_code_wmo_code_category",
]

# Pipeline de filtrage + caractéristiques temporelles
pipe_preproc = Pipeline([
    ("filter_columns", pps.ColumnFilterTransformer(columns_to_keep=keep_cols)),
    ("add_datetime_features", pps.DatetimePeriodicsTransformer(timestamp_col="date_et_heure_de_comptage")),
])

# Application du pipeline
df_preproc = pipe_preproc.fit_transform(df_cpt)

if df_preproc is not None and isinstance(df_preproc, pd.DataFrame):
    df = df_preproc.copy()
    print("✅ Pipeline appliqué.")
else:
    raise SystemExit("❌ Échec du prétraitement initial.")


In [ ]:
# === 3. Vérification / réinjection de la colonne datetime ===

if 'date_et_heure_de_comptage' not in df.columns:
    print("⚠️ Colonne 'date_et_heure_de_comptage' absente. Tentative de réinjection...")
    if len(df) == len(df_cpt):
        df['date_et_heure_de_comptage'] = df_cpt['date_et_heure_de_comptage'].values
        print("✅ Colonne 'date_et_heure_de_comptage' réinjectée avec succès.")
    else:
        raise ValueError("❌ Réinjection impossible : tailles incompatibles.")
else:
    print("✅ La colonne 'date_et_heure_de_comptage' est bien présente.")


In [ ]:
# === 4. Enrichissement avec features auto-régressives et moyennes mobiles ===

df_enriched = []

for site, df_site in df.groupby('nom_du_site_de_comptage'):
    df_site = df_site.sort_values(by='date_et_heure_de_comptage').copy()
    
    df_site['comptage_t-1'] = df_site['comptage_horaire'].shift(1)
    df_site['comptage_t-2'] = df_site['comptage_horaire'].shift(2)
    
    df_site['comptage_moy_3h'] = df_site['comptage_horaire'].rolling(window=3).mean()
    df_site['comptage_moy_6h'] = df_site['comptage_horaire'].rolling(window=6).mean()
    
    df_enriched.append(df_site)

df = pd.concat(df_enriched).dropna().reset_index(drop=True)
print(f"✅ Données enrichies. Nombre de lignes après traitement : {df.shape[0]}")



In [ ]:
# === 5. Détection des colonnes numériques / catégorielles + transformateurs ===

num_col = list(df.drop(columns='comptage_horaire').select_dtypes(include=np.number).columns)
cat_col = list(df.select_dtypes(include='object').columns)

tr_num_col = Pipeline(steps=[('standardisation', StandardScaler())])
tr_cat_col = Pipeline(steps=[('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))])

tr_columns = make_column_transformer(
    (tr_num_col, num_col),
    (tr_cat_col, cat_col)
)

print("✅ Transformateurs prêts.")
print(f" - Colonnes numériques : {len(num_col)}")
print(f" - Colonnes catégorielles : {len(cat_col)}")


In [ ]:
df_enriched = []

for site, df_site in df.groupby('nom_du_site_de_comptage'):
    df_site = df_site.sort_values(by='date_et_heure_de_comptage').copy()
    
    df_site['comptage_t-1'] = df_site['comptage_horaire'].shift(1)
    df_site['comptage_t-2'] = df_site['comptage_horaire'].shift(2)
    
    df_site['comptage_moy_3h'] = df_site['comptage_horaire'].rolling(window=3).mean()
    df_site['comptage_moy_6h'] = df_site['comptage_horaire'].rolling(window=6).mean()
    
    df_enriched.append(df_site)

df = pd.concat(df_enriched).dropna().reset_index(drop=True)
print(f"✅ Données enrichies : {df.shape[0]} lignes restantes après dropna.")


In [ ]:
num_col = list(df.drop(columns='comptage_horaire').select_dtypes(include=np.number).columns)
cat_col = list(df.select_dtypes(include='object').columns)
s_scaler = StandardScaler()
ohe_enc = OneHotEncoder(drop='first', handle_unknown='ignore')
tr_num_col = Pipeline(
    steps = [
        ('standardisation', s_scaler)
    ]
)
tr_cat_col = Pipeline(
    steps = [
        ('encoder', ohe_enc)
    ]
)
tr_columns = make_column_transformer( 
    (tr_num_col, num_col),
    (tr_cat_col, cat_col)
)

In [ ]:
df_enriched = []

for site, df_site in df.groupby('nom_du_site_de_comptage'):
    df_site = df_site.sort_values(by='date_et_heure_de_comptage').copy()
    
    df_site['comptage_t-1'] = df_site['comptage_horaire'].shift(1)
    df_site['comptage_t-2'] = df_site['comptage_horaire'].shift(2)
    
    df_site['comptage_moy_3h'] = df_site['comptage_horaire'].rolling(window=3).mean()
    df_site['comptage_moy_6h'] = df_site['comptage_horaire'].rolling(window=6).mean()
    
    df_enriched.append(df_site)

df = pd.concat(df_enriched).dropna().reset_index(drop=True)
print(f"✅ Données enrichies : {df.shape[0]} lignes restantes après dropna.")


# 3. Regression modeling

In [ ]:
def afficher_resultats_modele_temporel(compteur, model_results, periode_limite=('2025-04-01', '2025-04-16')):
    model = model_results[0]
    X_test_dates = model_results[1]
    y_test = model_results[2]
    y_test_pred = model_results[3]

    # Métriques
    r2 = r2_score(y_test, y_test_pred)
    rmse = root_mean_squared_error(y_test, y_test_pred)
    mae = mean_absolute_error(y_test, y_test_pred)

    # Affichage métriques
    print(f"\n📊 Résultats pour le compteur {compteur}")
    print(f"  - R²     : {r2:.4f}")
    print(f"  - RMSE   : {rmse:.2f}")
    print(f"  - MAE    : {mae:.2f}")
    print("-" * 40)

    # Tracé courbe
    plt.figure(figsize=(12, 8))
    plt.plot(X_test_dates.date_et_heure_de_comptage_local, y_test, label='Valeurs réelles')
    plt.plot(X_test_dates.date_et_heure_de_comptage_local, y_test_pred, label='Prédictions', linestyle='--')
    plt.title(f'Prédictions de la régression linéaire – Compteur {compteur}')
    plt.xlabel('Date')
    plt.xlim([pd.to_datetime(periode_limite[0]), pd.to_datetime(periode_limite[1])])
    plt.ylabel('Comptage Horaire')
    plt.legend()
    plt.grid(True)
    plt.show()

## 3.1 Linear Regresion

In [ ]:
pipe_linear_regression = Pipeline(
    steps= [
        ('preprocessing_column_transformation', tr_columns), 
        ('linear_regression_model',LinearRegression())
    ]
)

#### SANS AR(1) MA(24)

In [ ]:
models_lr_simple_results = {}
# pas d'aggrégation, juste un regroupement par nom de site et orientation (utilisé ensuite dans la boucle for)
grouped = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])
for compteur_id, df_compteur in grouped:
    # tri chrono + pipeline prétraitement + split
    df_compteur = df_compteur.sort_values("date_et_heure_de_comptage_local")
    X_train, X_train_dates, X_test, X_test_dates, y_train, y_test = \
        dfps.train_test_split_time_aware(
            df_compteur,
            timestamp_cols=["date_et_heure_de_comptage_utc", "date_et_heure_de_comptage_local"],
            target_col="comptage_horaire"
        )
    # pipeline + fit
    model = pipe_linear_regression.fit(X_train, y_train)
    # ohe = preprocessor.named_transformers_['cat']
    # print("Colonnes encodées :", preprocessor.transformers_[0][2])
    y_test_pred = model.predict(X_test)
    models_lr_simple_results[compteur_id] = [model, X_test_dates, y_test, y_test_pred]

for key in models_lr_simple_results:
    print(f"- {key}")

In [ ]:
# Limiter l'affichage/analyse à un sous-ensemble de compteurs
nb_compteurs = 2
for compteur, model_results in islice(models_lr_simple_results.items(), nb_compteurs):
    afficher_resultats_modele_temporel(compteur, model_results)

#### AVEC AR(1) MA(24)

## 3.1 KNN Regresion

In [ ]:
pipe_knn_regression = Pipeline(
    steps= [
        ('preprocessing_column_transformation', tr_columns), 
        ('knn_regression_model',KNeighborsRegressor(n_jobs=-1, metric='minkowski'))
    ]
)

#### SANS AR(1) MA(24)

In [ ]:
models_knn_simple_results = {}
# pas d'aggrégation, juste un regroupement par nom de site et orientation (utilisé ensuite dans la boucle for)
grouped = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])
for compteur_id, df_compteur in grouped:
    # tri chrono + pipeline prétraitement + split
    df_compteur = df_compteur.sort_values("date_et_heure_de_comptage_local")
    X_train, X_train_dates, X_test, X_test_dates, y_train, y_test = \
        dfps.train_test_split_time_aware(
            df_compteur,
            timestamp_cols=["date_et_heure_de_comptage_utc", "date_et_heure_de_comptage_local"],
            target_col="comptage_horaire"
        )
    # pipeline + fit
    model = pipe_knn_regression.fit(X_train, y_train)
    y_test_pred = model.predict(X_test)
    models_knn_simple_results[compteur_id] = [model, X_test_dates, y_test, y_test_pred]

for key in models_knn_simple_results:
    print(f"- {key}")

In [ ]:
# Limiter l'affichage/analyse à un sous-ensemble de compteurs
nb_compteurs = 2
for compteur, model_results in islice(models_knn_simple_results.items(), nb_compteurs):
    afficher_resultats_modele_temporel(compteur, model_results)

#### AVEC AR(1) MA(24)

## Régression Random Forest (ajoutée)

Ce modèle est ajouté à titre comparatif avec les mêmes variables et prétraitements que les autres modèles du notebook.

In [ ]:

from sklearn.ensemble import RandomForestRegressor

print("\n--- Régression Random Forest ---")
pipe_random_forest_regression = Pipeline([
    ('transformation', tr_columns),
    ('modele', RandomForestRegressor(
        random_state=210995,
        n_jobs=-1,
        n_estimators=100,
        max_depth=25,
        min_samples_leaf=5,
        max_features='sqrt'
    ))
])

pipe_random_forest_regression.fit(X_train, y_train)
y_test_pred_rf = pipe_random_forest_regression.predict(X_test)

print("R² :", r2_score(y_test, y_test_pred_rf))
print("Erreur Absolue Moyenne :", mean_absolute_error(y_test, y_test_pred_rf))


In [ ]:
# 2. Réel vs Prédit
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_test_pred_rf, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r')
plt.xlabel("Valeurs réelles")
plt.ylabel("Prédictions")
plt.title("Random Forest : Réel vs Prédit")
plt.grid(True)
plt.show()


In [ ]:
#3. Distribution des résidus
residuals_rf = y_test - y_test_pred_rf
plt.figure(figsize=(8, 4))
sns.histplot(residuals_rf, bins=50, kde=True)
plt.title("Distribution des résidus (Random Forest)")
plt.xlabel("Erreur de prédiction")
plt.grid(True)
plt.show()


In [ ]:
#4. Résidus vs Prédictions
plt.figure(figsize=(8, 5))
plt.scatter(y_test_pred_rf, residuals_rf, alpha=0.3)
plt.axhline(0, linestyle='--', color='red')
plt.xlabel("Valeurs prédites")
plt.ylabel("Résidus")
plt.title("Résidus vs Prédictions - Random Forest")
plt.grid(True)
plt.show()

In [ ]:
#Erreurs absolues triées 
errors = np.abs(y_test - y_test_pred_rf)
sorted_errors = np.sort(errors)

plt.figure(figsize=(10, 4))
plt.plot(sorted_errors)
plt.xlabel("Échantillons test triés")
plt.ylabel("Erreur absolue")
plt.title("Erreurs absolues triées - Random Forest")
plt.grid(True)
plt.show()

In [ ]:
#Cette fois il faudra visualiser par compteur et rajouter du gridsearch pour optimiser les hyperparamètres du modèle Random Forest. 
#Random Forest il faudra rajouter la répartion globale et prendre la dimension temporelle en compte. 
#Basculons sur des compteurs unitaires en Random Forest, avec une répartition temporelle.

In [ ]:
#Cette fois il faudra visualiser par compteur et rajouter du gridsearch pour optimiser les hyperparamètres du modèle Random Forest. 
#Random Forest il faudra rajouter la répartion globale et prendre la dimension temporelle en compte. 
#Basculons sur des compteurs unitaires en Random Forest, avec une répartition temporelle.



In [ ]:
# 2. Conversion datetime avec gestion du fuseau horaire (rapide et efficace)
import pandas as pd
df_cpt['date_et_heure_de_comptage'] = pd.to_datetime(df_cpt['date_et_heure_de_comptage'], utc=True)

# 3. Vérification rapide
print(df_cpt['date_et_heure_de_comptage'].head())

In [ ]:
import smartcheck.preprocessing_project_specific as pps
from sklearn.pipeline import Pipeline

# Colonnes à conserver
keep_cols = [
    "nom_du_site_de_comptage",
    "comptage_horaire",
    "date_et_heure_de_comptage",
    "orientation_compteur",
    "latitude", "longitude",
    "arrondissement",
    "jour_ferie", "vacances_scolaires",
    "temperature_2m_c", "rain_mm", "snowfall_cm",
    "elevation", "weather_code_wmo_code_category",
]

pipe_preproc = Pipeline([
    ("filter_columns", pps.ColumnFilterTransformer(columns_to_keep=keep_cols)),
    ("add_datetime_features", pps.DatetimePeriodicsTransformer(timestamp_col="date_et_heure_de_comptage")),
])

df_preproc = pipe_preproc.fit_transform(df_cpt)
df = df_preproc.copy()


In [ ]:
# 2. Conversion datetime avec gestion du fuseau horaire (rapide et efficace)
import pandas as pd
df_cpt['date_et_heure_de_comptage'] = pd.to_datetime(df_cpt['date_et_heure_de_comptage'], utc=True)

# 3. Vérification rapide
print(df_cpt['date_et_heure_de_comptage'].head())

In [ ]:
# Choisissez un identifiant de compteur à visualiser
compteur_exemple = df['nom_du_site_de_comptage'].unique()[0]

df_plot = df[df['nom_du_site_de_comptage'] == compteur_exemple]

plt.figure(figsize=(14, 4))
plt.plot(df_plot['date_et_heure_de_comptage_utc'], df_plot['comptage_horaire'], label='Comptage horaire')
plt.title(f"Comptage horaire dans le temps - Compteur : {compteur_exemple}")
plt.xlabel("Date et heure")
plt.ylabel("Comptage")
plt.grid(True)
plt.tight_layout()
plt.legend()
plt.show()


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

# Données pour le compteur choisi
df_unit = df[df['nom_du_site_de_comptage'] == compteur_exemple].copy()

# Définir les colonnes
target = 'comptage_horaire'
features_to_exclude = ['nom_du_site_de_comptage', 'date_et_heure_de_comptage_utc', target]
X = df_unit.drop(columns=features_to_exclude)
y = df_unit[target]

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Transformation colonnes numériques / catégorielles
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

preprocessor = make_column_transformer(
    (StandardScaler(), num_cols),
    (OneHotEncoder(handle_unknown='ignore'), cat_cols)
)

pipe_rf = Pipeline([
    ('preprocessing', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

pipe_rf.fit(X_train, y_train)
y_pred = pipe_rf.predict(X_test)

print("R² :", r2_score(y_test, y_pred))
print("MAE :", mean_absolute_error(y_test, y_pred))


In [ ]:
from sklearn.model_selection import GridSearchCV

# Grille de recherche
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [10, 20, None],
    'model__min_samples_leaf': [1, 3, 5],
    'model__max_features': ['sqrt', 'log2']
}

grid_search = GridSearchCV(
    pipe_rf,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# Évaluation
best_model = grid_search.best_estimator_
y_pred_grid = best_model.predict(X_test)

print("Meilleurs paramètres trouvés :", grid_search.best_params_)
print("R² (optimisé) :", r2_score(y_test, y_pred_grid))
print("MAE (optimisé) :", mean_absolute_error(y_test, y_pred_grid))


In [ ]:
# Visualisation Réel vs Prédit
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred_grid, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r')
plt.xlabel("Valeurs réelles")
plt.ylabel("Valeurs prédites")
plt.title(f"Réel vs Prédit - Compteur : {compteur_exemple}")


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

resultats = []

compteurs = df['nom_du_site_de_comptage'].unique()

for compteur_id in compteurs:
    df_unit = df[df['nom_du_site_de_comptage'] == compteur_id].copy()
    
    if len(df_unit) < 100:  # pour éviter les échantillons trop petits
        continue

    X = df_unit.drop(columns=['nom_du_site_de_comptage', 'date_et_heure_de_comptage_utc', 'comptage_horaire'])
    y = df_unit['comptage_horaire']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    num_cols = X.select_dtypes(include='number').columns.tolist()
    cat_cols = X.select_dtypes(include='object').columns.tolist()

    preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])
    

    pipe_rf = Pipeline([
        ('preprocessing', preprocessor),
        ('model', RandomForestRegressor(random_state=42))
    ])

    # Pour accélérer, on peut faire un mini-GridSearch rapide
    param_grid = {
        'model__n_estimators': [100],
        'model__max_depth': [10],
        'model__min_samples_leaf': [3],
        'model__max_features': ['sqrt']
    }

    grid_search = GridSearchCV(
        pipe_rf,
        param_grid=param_grid,
        cv=3,
        scoring='r2',
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)
    y_pred = grid_search.best_estimator_.predict(X_test)

    resultats.append({
        'compteur': compteur_id,
        'r2': r2_score(y_test, y_pred),
        'mae': mean_absolute_error(y_test, y_pred),
        'params': grid_search.best_params_
    })

# Résumé sous forme de DataFrame
df_resultats = pd.DataFrame(resultats).sort_values(by='r2', ascending=False)
display(df_resultats.head(10))


# 4. Régression linéaire temporelle
Nous allons effectuer une régression linéaire sur les données de 5 compteurs vélo, en prenant en compte des variables temporelles et météorologiques.

In [ ]:
print(df["date_et_heure_de_comptage"].head())
print(df["date_et_heure_de_comptage"].dtype)


In [ ]:
# Forcer la conversion en datetime
df["date_et_heure_de_comptage_utc"] = pd.to_datetime(df["date_et_heure_de_comptage_utc"], errors="coerce", utc=True)

# Supprimer la timezone (si nécessaire pour éviter les erreurs .dt)
df["date_et_heure_de_comptage_utc"] = df["date_et_heure_de_comptage_utc"].dt.tz_convert(None)

# Maintenant on peut extraire les composantes temporelles
df["hour"] = df["date_et_heure_de_comptage"].dt.hour
df["dayofweek"] = df["date_et_heure_de_comptage"].dt.dayofweek
df["month"] = df["date_et_heure_de_comptage"].dt.month




In [ ]:

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

# Charger la base complète
df = pd.read_csv("comptage-velo-donnees-compteurs-2024-2025_Enriched_ML-ready_data (1).csv", parse_dates=["date_et_heure_de_comptage"])

# Création des variables temporelles
df["hour"] = df["date_et_heure_de_comptage"].dt.hour
df["dayofweek"] = df["date_et_heure_de_comptage"].dt.dayofweek
df["month"] = df["date_et_heure_de_comptage"].dt.month

# Choix de 5 compteurs aléatoires
top5_counters = df["identifiant_du_compteur"].value_counts().index[:5]

for counter in top5_counters:
    df_counter = df[df["identifiant_du_compteur"] == counter].copy()

    features = ["hour", "dayofweek", "month", "temperature_2m_c", "rain_mm", "snowfall_cm"]
    df_counter.dropna(subset=features + ["comptage_horaire"], inplace=True)

    X = df_counter[features]
    y = df_counter["comptage_horaire"]

    model = LinearRegression()
    model.fit(X, y)
    y_pred = model.predict(X)

    print(f"Compteur : {counter}")
    print("RMSE :", mean_squared_error(y, y_pred, squared=False))

    plt.figure(figsize=(10, 4))
    plt.plot(df_counter["date_et_heure_de_comptage"], y, label="Réel", alpha=0.6)
    plt.plot(df_counter["date_et_heure_de_comptage"], y_pred, label="Prédit", alpha=0.6)
    plt.title(f"Régression linéaire - {counter}")
    plt.xlabel("Date")
    plt.ylabel("Comptage")
    plt.legend()
    plt.tight_layout()
    plt.show()


# 5. Random Forest par compteur avec GridSearchCV
Nous allons entraîner un modèle Random Forest pour chaque compteur de manière unitaire, avec GridSearchCV pour optimiser les hyperparamètres. Les variables temporelles seront intégrées.

In [ ]:
import pandas as pd

# Charger les données
df = pd.read_csv("comptage-velo-donnees-compteurs-2024-2025_Enriched_ML-ready_data (1).csv")

# 🔧 1. Nettoyage : suppression des éventuels espaces, conversion forcée
df["date_et_heure_de_comptage"] = pd.to_datetime(
    df["date_et_heure_de_comptage"].astype(str).str.strip(),
    errors="coerce",
    utc=True  # ajoute UTC pour contourner les problèmes de timezone
)

# 🔁 2. Supprimer la timezone (nécessaire pour `.dt`)
df["date_et_heure_de_comptage"] = df["date_et_heure_de_comptage"].dt.tz_convert(None)

# 🔍 3. Vérification
print("Type :", df["date_et_heure_de_comptage"].dtype)
print("Valeurs manquantes :", df["date_et_heure_de_comptage"].isna().sum())

# ✅ 4. Extraction des variables temporelles
df["hour"] = df["date_et_heure_de_comptage"].dt.hour
df["dayofweek"] = df["date_et_heure_de_comptage"].dt.dayofweek
df["month"] = df["date_et_heure_de_comptage"].dt.month
df["dayofyear"] = df["date_et_heure_de_comptage"].dt.dayofyear


In [ ]:
df_counter["date"] = df_counter["date_et_heure_de_comptage"].dt.date
df_daily = df_counter.groupby("date")[["y_true", "y_pred"]].mean().reset_index()

plt.figure(figsize=(12, 5))
plt.plot(df_daily["date"], df_daily["y_true"], label="Réel (moy. quotidienne)")
plt.plot(df_daily["date"], df_daily["y_pred"], label="Prédit (moy. quotidienne)")
plt.title(f"Random Forest - {site}")
plt.xlabel("Date")
plt.ylabel("Comptage journalier moyen")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from math import sqrt
# Prise en compte du temps
df["dayofyear"] = df["date_et_heure_de_comptage"].dt.dayofyear

# Paramètres pour GridSearchCV
param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [5, 10]
}

tscv = TimeSeriesSplit(n_splits=3)
top5_counters = df["nom_du_site_de_comptage"].value_counts().index[:5]
for counter in top5_counters:
    df_counter = df[df["nom_du_site_de_comptage"] == counter].copy()
    features = ["hour", "dayofweek", "month", "dayofyear", "temperature_2m_c", "rain_mm", "snowfall_cm"]
    df_counter.dropna(subset=features + ["comptage_horaire"], inplace=True)

    X = df_counter[features]
    y = df_counter["comptage_horaire"]
    

    rf = RandomForestRegressor(random_state=42)
    grid = GridSearchCV(rf, param_grid, cv=tscv, scoring="neg_root_mean_squared_error", n_jobs=-1)
    grid.fit(X, y)
    best_rf = grid.best_estimator_

    y_pred = best_rf.predict(X)

    print(f"Compteur : {counter}")
    print("Best Params:", grid.best_params_)
    print("Best RMSE :", sqrt(mean_squared_error(y, y_pred)))

    plt.figure(figsize=(10, 4))
    plt.plot(df_counter["date_et_heure_de_comptage"], y, label="Réel", alpha=0.6)
    plt.plot(df_counter["date_et_heure_de_comptage"], y_pred, label="Prédit", alpha=0.6)
    plt.title(f"Random Forest - {counter}")
    plt.xlabel("Date")
    plt.ylabel("Comptage")
    plt.legend()
    plt.tight_layout()
    plt.show()


### 🔄 Chargement et traitement des données temporelles

In [ ]:

import pandas as pd

df = pd.read_csv("comptage-velo-donnees-compteurs-2024-2025_Enriched_ML-ready_data (1).csv")

df["date_et_heure_de_comptage"] = pd.to_datetime(
    df["date_et_heure_de_comptage"].astype(str).str.strip(),
    errors="coerce",
    utc=True
)
df["date_et_heure_de_comptage"] = df["date_et_heure_de_comptage"].dt.tz_convert(None)

df["hour"] = df["date_et_heure_de_comptage"].dt.hour
df["dayofweek"] = df["date_et_heure_de_comptage"].dt.dayofweek
df["month"] = df["date_et_heure_de_comptage"].dt.month
df["dayofyear"] = df["date_et_heure_de_comptage"].dt.dayofyear


### 🌲 Random Forest par compteur avec GridSearchCV (visualisation lissée)

In [ ]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from math import sqrt
import matplotlib.pyplot as plt

top5_counters = df["identifiant_du_compteur"].value_counts().index[:5]

param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [5, 10]
}
tscv = TimeSeriesSplit(n_splits=3)

for counter in top5_counters:
    df_counter = df[df["identifiant_du_compteur"] == counter].copy()
    site = df_counter["nom_du_site_de_comptage"].iloc[0]

    features = ["hour", "dayofweek", "month", "dayofyear", "temperature_2m_c", "rain_mm", "snowfall_cm"]
    df_counter.dropna(subset=features + ["comptage_horaire"], inplace=True)

    X = df_counter[features]
    y = df_counter["comptage_horaire"]

    rf = RandomForestRegressor(random_state=42)
    grid = GridSearchCV(rf, param_grid, cv=tscv, scoring="neg_root_mean_squared_error", n_jobs=-1)
    grid.fit(X, y)
    best_rf = grid.best_estimator_
    y_pred = best_rf.predict(X)

    rmse = sqrt(mean_squared_error(y, y_pred))
    print(f"Compteur : {counter} ({site})")
    print("Best Params:", grid.best_params_)
    print("Best RMSE :", rmse)

    df_counter["y_true"] = y.values
    df_counter["y_pred"] = y_pred
    df_counter["y_true_smooth"] = df_counter["y_true"].rolling(window=24).mean()
    df_counter["y_pred_smooth"] = df_counter["y_pred"].rolling(window=24).mean()

    plt.figure(figsize=(12, 5))
    plt.plot(df_counter["date_et_heure_de_comptage"], df_counter["y_true_smooth"], label="Réel (lissé)")
    plt.plot(df_counter["date_et_heure_de_comptage"], df_counter["y_pred_smooth"], label="Prédit (lissé)")
    plt.title(f"Random Forest - {site}")
    plt.xlabel("Date")
    plt.ylabel("Comptage horaire (moy. mobile)")
    plt.legend()
    plt.tight_layout()
    plt.show()


### 📈 Régressions linéaires temporelles pour 4 compteurs sélectionnés

In [ ]:

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from math import sqrt
import matplotlib.pyplot as plt

compteurs_rl = [
    "Pont de Bercy",
    "135 avenue Daumesnil",
    "180 avenue d'Italie",
    "27 quai de la Tournelle"
]

rl_results = []

for site in compteurs_rl:
    df_site = df[df["nom_du_site_de_comptage"] == site].dropna()
    features = ["hour", "dayofweek", "month", "dayofyear", "temperature_2m_c", "rain_mm", "snowfall_cm"]

    X = df_site[features]
    y = df_site["comptage_horaire"]

    model = LinearRegression()
    model.fit(X, y)
    y_pred = model.predict(X)
    rmse = sqrt(mean_squared_error(y, y_pred))
    rl_results.append((site, rmse))

    df_site["y_true"] = y.values
    df_site["y_pred"] = y_pred

    df_daily = df_site.groupby(df_site["date_et_heure_de_comptage"].dt.date)[["y_true", "y_pred"]].mean().reset_index()

    plt.figure(figsize=(12, 5))
    plt.plot(df_daily["date_et_heure_de_comptage"], df_daily["y_true"], label="Réel (moyenne journalière)")
    plt.plot(df_daily["date_et_heure_de_comptage"], df_daily["y_pred"], label="Prédit (moyenne journalière)")
    plt.title(f"Régression Linéaire - {site}")
    plt.xlabel("Date")
    plt.ylabel("Comptage moyen")
    plt.legend()
    plt.tight_layout()
    plt.show()

print("Résultats régression linéaire :")
for site, rmse in rl_results:
    print(f"{site} : RMSE = {rmse:.2f}")
